## Projekt - Baza Danych dla Systemu Piekielnego

Autor: Justyna Nowosad

## 1. Tworzenie struktury bazy danych.

Tworzenie bazy danych opiera się na wcześniej stworzonych diagramach: konceptualnym i implementacyjnym, stworzonych za pomocą programu Vertabelo. Najważniejszym zadaniem tej sekcji było dopilnowanie, aby tabele były tworzone w odpowiedniej kolejności, dlatego zaczęłam od tabel niezależnych (np. grzech czy kara, które są swego rodzaju "słownikami"), następnie przeszłam do podstaowych encji, które stanowią bazę całego systemu piekielnego i będą stale używane - grzesznik i diabeł. Na koniec zostawiłam wszystkie tabele, które wykazują relacje z encjami podstawowymi. Dzięki temu miałam pewność, że Foreign Key nie odwołuje się do żadnych tabel, które nie istnieją. 

In [96]:
-- STWORZENIE TABEL NIEZALEŻNYCH , KTÓRE NIE POSIADAJĄ KLUCZY OBCYCH 

CREATE TABLE Funkcja_diabla
(
    ID_funkcja INT PRIMARY KEY IDENTITY(1,1),
    nazwa NVARCHAR(50) NOT NULL
);

CREATE TABLE Kara
(
    ID_kara INT PRIMARY KEY IDENTITY(1,1),
    rodzaj_kary NVARCHAR(50) NOT NULL
);

CREATE TABLE Grzech
(
    ID_grzech INT PRIMARY KEY IDENTITY(1,1),
    nazwa NVARCHAR(50) NOT NULL,
    kategoria NVARCHAR(50) NOT NULL CHECK (kategoria IN (N'Lekki', N'Średni', N'Ciężki')), -- Założyłam trzy dopuszczalne kategorie grzechów
    skala_max INT NOT NULL
);

CREATE TABLE Osoba_zyjaca
(
    ID_osoba INT PRIMARY KEY IDENTITY(1,1),
    imie NVARCHAR(50) NOT NULL,
    nazwisko NVARCHAR(50) NOT NULL,
    czy_zyje BIT NOT NULL -- Chociaż pytanie, czy osoba żyjąca nadal żyje wydaje się dosyć abtrakcyjne, to ta kolumna ułatwi pózniejsze sprawdzanie, który gorliwy powinien być celem kuszenia diabła, bo może zdarzyć się, że osoba, która najczęściej się modliła już nie żyje 
);

CREATE TABLE Rada_piekielna
(
    ID_rada INT PRIMARY KEY IDENTITY(1,1),
    kadencja_od DATE NOT NULL, -- Rozważałam użycie liczbowego typu danych, aby móc obsługiwać lata powyżej 9999, ale uniemożliwiłoby to korzystanie z funkcji czasu np. DATEDIFF, dlatego zdecydowałam, że w tym przypadku poprawność obliczeń jest ważniejsza niż bardzo duży zakres czasu
    kadencja_do DATE NULL -- Jeśli tutaj występuje NULL, to znaczy, że kadencja trwa
);

-- STWORZENIE PODSTAWOWYCH ENCJI - GRZESZNIK I DIABEŁ

CREATE TABLE Grzesznik
(
    ID_grzesznik INT PRIMARY KEY IDENTITY(1,1),
    imie NVARCHAR(50) NOT NULL,
    nazwisko NVARCHAR(50) NOT NULL,
    data_smierci DATE NOT NULL,
    data_urodzenia DATE NOT NULL, -- Dodałam datę urodzenia, aby Rada Piekielna mogła obliczać częstotliwość grzeszenia przypadającą na rok życia grzesznika, uznałam to za sprawiedliwy sposób na ocenianie częstotliwości
    status NVARCHAR(15) DEFAULT 'Cierpiący' CHECK (status IN (N'Cierpiący', N'Awansowany')) -- Każdy grzesznik może dostać karę lub być awansowanym na diabła
);

CREATE TABLE Diabel
(
    ID_diabel INT PRIMARY KEY IDENTITY(1,1),
    nazwa NVARCHAR(50) NOT NULL,
    data_powolania DATE NOT NULL,
    status NVARCHAR(20) NOT NULL CHECK (status IN (N'Zwykły', N'Członek rady', N'Doradca Lucyfera')), -- Diabeł może mieć jeden z trzech określonych statusów, Lucyfer, jako król piekła, nie jest liczony jako przeciętny diabeł, on jest niezmiennym bytem niezależnym
    diabel_ID_grzesznik INT NULL, -- Może być NULL, jeśli diabeł nie pochodzi od grzesznika np. jest pradawnym demonem
    FOREIGN KEY (diabel_ID_grzesznik) REFERENCES Grzesznik(ID_grzesznik)
);

-- STWORZENIE TABEL RELACYJNYCH

CREATE TABLE Przydzial_funkcji
(
    ID_przydzial INT PRIMARY KEY IDENTITY(1,1),
    data_od DATE NOT NULL,
    data_do DATE NULL,
    ocena_pracy INT CHECK (ocena_pracy BETWEEN 1 AND 10), -- Ustawiam prostą ocenę pracy w skali 1-10
    przydzial_ID_diabel INT NOT NULL,
    przydzial_ID_funkcja INT NOT NULL,
    FOREIGN KEY (przydzial_ID_diabel) REFERENCES Diabel(ID_diabel),
    FOREIGN KEY (przydzial_ID_funkcja) REFERENCES Funkcja_diabla(ID_funkcja)
);

CREATE TABLE Czlonek_rady
(
    ID_czlonek INT PRIMARY KEY IDENTITY(1,1),
    powod_wyboru NVARCHAR(100) NOT NULL,
    czlonek_ID_rada INT NOT NULL,
    czlonek_ID_diabel INT NOT NULL,
    FOREIGN KEY (czlonek_ID_rada) REFERENCES Rada_piekielna(ID_rada),
    FOREIGN KEY (czlonek_ID_diabel) REFERENCES Diabel(ID_diabel)
);

CREATE TABLE Doradca_lucyfera
(
    ID_doradca INT PRIMARY KEY IDENTITY(1,1),
    sfera_dzialania NVARCHAR(100) NOT NULL,
    data_nominacji DATE NOT NULL,
    doradca_ID_diabel INT NOT NULL UNIQUE, -- Każdy diabeł może zostać doradcą raz, dlatego UNIQUE
    doradca_ID_czlonek INT NOT NULL UNIQUE, -- Diabeł zostaje doradcą Lucyfera po wygaśnięciu rady, której był członkiem 
    FOREIGN KEY (doradca_ID_diabel) REFERENCES Diabel(ID_diabel),
    FOREIGN KEY (doradca_ID_czlonek) REFERENCES Czlonek_rady(ID_czlonek)
);

CREATE TABLE Wpis_grzech
(
    ID_wpis INT PRIMARY KEY IDENTITY(1,1),
    wpis_ID_grzesznik INT NOT NULL,
    wpis_ID_grzech INT NOT NULL,
    ocena_grzechu INT NOT NULL,
    data_popelnienia DATE NOT NULL,
    FOREIGN KEY (wpis_ID_grzesznik) REFERENCES Grzesznik(ID_grzesznik),
    FOREIGN KEY (wpis_ID_grzech) REFERENCES Grzech(ID_grzech)
);

CREATE TABLE Modlitwa
(
    ID_modlitwa INT PRIMARY KEY IDENTITY(1,1),
    data DATE NOT NULL,
    modlitwa_ID_osobaZyjaca INT NOT NULL,
    modlitwa_ID_grzesznik INT NOT NULL,
    FOREIGN KEY (modlitwa_ID_osobaZyjaca) REFERENCES Osoba_zyjaca(ID_osoba),
    FOREIGN KEY (modlitwa_ID_grzesznik) REFERENCES Grzesznik(ID_grzesznik)
);

CREATE TABLE Kuszenie
(
    ID_kuszenie INT PRIMARY KEY IDENTITY(1,1),
    data DATE NOT NULL,
    skutecznosc BIT NOT NULL,
    kuszenie_ID_grzech INT NOT NULL,
    kuszenie_ID_diabel INT NOT NULL,
    kuszenie_ID_osoba INT NOT NULL,
    FOREIGN KEY (kuszenie_ID_grzech) REFERENCES Grzech(ID_grzech), -- Surowość grzechu będzie pobierana z tabeli grzech, używając ID_grzech, dlatego błędem byłoby powielać ją również w tej tabeli
    FOREIGN KEY (kuszenie_ID_diabel) REFERENCES Diabel(ID_diabel),
    FOREIGN KEY (kuszenie_ID_osoba) REFERENCES Osoba_zyjaca(ID_osoba)
);

CREATE TABLE Wymierzona_kara
(
    ID_wymierzona_kara INT PRIMARY KEY IDENTITY(1,1),
    srednia_ocena_grzechow DECIMAL(10,2) NOT NULL,
    czestotliwosc_grzechow DECIMAL(10,2) NOT NULL,
    surowosc DECIMAL(6,2) NOT NULL CHECK (surowosc >=0 AND surowosc <= 1000), -- Skala surowości kary jest z zakesu 0-1000, type DECIMAL, ponieważ w moim systemie modlitwa odejmuje 0.5 od surowości 
    data_od DATE NOT NULL,
    data_do DATE NULL, -- NULL oznacza karę wieczną
    kara_ID_kara INT NOT NULL,
    kara_ID_grzesznik INT NOT NULL UNIQUE, -- Jeden grzesznik ma jedną wymierzoną karę
    kara_ID_rada INT NOT NULL,
    FOREIGN KEY (kara_ID_kara) REFERENCES Kara(ID_kara),
    FOREIGN KEY (kara_ID_grzesznik) REFERENCES Grzesznik(ID_grzesznik),
    FOREIGN KEY (kara_ID_rada) REFERENCES Rada_piekielna(ID_rada)
);

CREATE TABLE Nadzor_kary
(
    ID_nadzor INT PRIMARY KEY IDENTITY(1,1),
    data_od DATE NOT NULL,
    data_do DATE NULL,
    nadzor_ID_diabel INT NOT NULL,
    nadzor_ID_kara INT NOT NULL,
    FOREIGN KEY (nadzor_ID_diabel) REFERENCES Diabel(ID_diabel),
    FOREIGN KEY (nadzor_ID_kara) REFERENCES Wymierzona_kara(ID_wymierzona_kara)
);

CREATE TABLE Raport
(
    ID_raport INT PRIMARY KEY IDENTITY(1,1),
    data DATE NOT NULL,
    liczba_diablow_lacznie INT NOT NULL,
    liczba_diablow_zwyklych INT NOT NULL,
    liczba_czlonkow_rady INT NOT NULL,
    liczba_doradcow INT NOT NULL,
    liczba_grzesznikow INT NOT NULL,
    zmiana_diablow INT NOT NULL, -- Raporty tratuję jako historię danych o piekle, wiem, że zmianę można policzyć, ale jeśli w przyszłości piekła pytanie biznesowe będzie opierało się o identyfikację lat z największym spadkiem/przyrostem, to najprościej przechowywać te dane tutaj
    zmiana_grzesznikow INT NOT NULL,
    raport_ID_rada INT NOT NULL,
    FOREIGN KEY (raport_ID_rada) REFERENCES Rada_piekielna(ID_rada)
);

Commands completed successfully.

Total execution time: 00:00:00.167

W celu zapewnienia płynności działania systemu dodałam indeksy nieklastrowe na kolumnach, które w bazie danych są najczęściej wykorzystywane w warunkach JOIN czy WHERE. Indeksy zostały dobrane na podstawie przewidywanych zapytań biznesowych i procedur składowanych. Zastosowanie indeksów ogranicza konieczność pełnego przeszukiwania tabel, dzięki czemu zwiększa szybkość i wydajność przetwarzania zapytań. 

In [97]:
-- INDEKS UNIKALNY NA POLU diabeł_ID_grzesznik
CREATE UNIQUE INDEX INDEX_Diabel_UnikalnyGrzesznik 
ON Diabel(diabel_ID_grzesznik) 
WHERE diabel_ID_grzesznik IS NOT NULL;

-- INDEKS DO PROSTSZEGO ZLICZANIA GRZECHÓW
CREATE INDEX INDEX_WpisGrzech_Grzesznik 
ON Wpis_grzech(wpis_ID_grzesznik);

-- INDEKS DO KONTROLI KUSZENIA
CREATE INDEX INDEX_Kuszenie_Diabel 
ON Kuszenie(kuszenie_ID_diabel);

CREATE INDEX INDEX_Kuszenie_Osoba
ON Kuszenie(kuszenie_ID_osoba);

-- INDEKS DO KONTROLI MODLITW
CREATE INDEX INDEX_Modlitwa_Osoba
ON Modlitwa(modlitwa_ID_osobaZyjaca);

CREATE INDEX INDEX_Modlitwa_Grzesznik
ON Modlitwa(modlitwa_ID_grzesznik);

-- INDEKS DO PRZYSPIESZANIA SPRAWDZANIA LIMITU 1000 NADZORÓW KAR NA DIABŁA (W SEKCJI Z PROCEDURAMI)
CREATE INDEX INDEX_NadzorKary_Diabel
ON Nadzor_kary(nadzor_ID_diabel);

-- INDEKS DO USPRAWNIENIA SPRAWDZANIA, ILU JEST GRZESZNIKÓW Z AKTYWNĄ KARĄ
CREATE INDEX INDEX_WymierzonaKara_DataDo 
ON Wymierzona_kara(data_do);

Commands completed successfully.

Total execution time: 00:00:00.088

## 2\. Wprowadzanie danych testowych.

Na potrzeby projektu wprowadziłam do każdej tabeli dane testowe. Ich liczba jest stosunkowo mała w pórównaniu do danych, które realnie znalazłyby się w tego typu bazie danych, ale wystarczająca, aby przetestować zapytania w kolejnej części projektu. Jako pierwsze wprowadzam dane ustalone z góry, takie jak dostępne grzechy, kary, funkcje diabłów.

In [98]:
INSERT INTO Grzech
    (nazwa, kategoria, skala_max)
VALUES
    (N'Plotkowanie', N'Lekki', 20),
    (N'Spóźnialstwo', N'Lekki', 20),
    (N'Drobne kłamstwo', N'Lekki', 20),
    (N'Bałaganiarstwo', N'Lekki', 20),
    (N'Narzekanie', N'Lekki', 20),
    (N'Kradzież - mała wartość', N'Średni', 50),
    (N'Zazdrość', N'Średni', 50),
    (N'Oszustwo', N'Średni', 50),
    (N'Pycha', N'Średni', 50),
    (N'Obrażanie innych', N'Średni', 50),
    (N'Morderstwo', N'Ciężki', 100),
    (N'Przemoc fizyczna', N'Ciężki', 100),
    (N'Niszczenie mienia', N'Ciężki', 100),
    (N'Kradzież - duża wartość', N'Ciężki', 100),
    (N'Prześladowanie', N'Ciężki', 100);

INSERT INTO Kara
    (rodzaj_kary)
VALUES
    (N'Gotowanie w smole'),
    (N'Biczowanie rażącymi łańcuchami'),
    (N'Obdzieranie ze skóry'),
    (N'Wtaczanie głazu pod górę'),
    (N'Liczenie ziaren piasku'),
    (N'Karmienie gorącymi węglami'),
    (N'Rozciąganie na kole tortur'),
    (N'Zamrożenie w lodowym jeziorze'),
    (N'Wyrywanie jezyka'),
    (N'Bezgraniczna rozpacz');

INSERT INTO Funkcja_diabla
    (nazwa)
VALUES
    (N'Kusiciel'),
    (N'Dręczyciel');

INSERT INTO Diabel
    (nazwa, data_powolania, status, diabel_ID_grzesznik) -- Dodanie podstawowych demonów niepochodzących od grzeszników
VALUES
    (N'Belzebub', '0006-06-06', N'Zwykły', NULL),
    (N'Asmodeusz', '0066-06-06', N'Zwykły', NULL),
    (N'Mammon', '0666-06-06', N'Zwykły', NULL),
    (N'Mefistofeles', '0606-06-06', N'Zwykły', NULL),
    (N'Astaroth', '0660-06-06', N'Zwykły', NULL);

(15 rows affected)

(10 rows affected)

(2 rows affected)

(5 rows affected)

Total execution time: 00:00:00.065

Następnie dodałam dane wygenerowane przez beekeeperstudio.io oraz inne dane losowe, które zostały wprowadzone zarówno ręcznie, jak i przy użyciu mechanizmów generowania danych w języku SQL (np. CROSS JOIN). Dane zostały przygotowane tak, aby zapewnić spójność pomiędzy tabelami w celu realizacji zapytań biznesowych wymaganych w dalszej części projektu. 

In [99]:
INSERT INTO Osoba_zyjaca
    (imie, nazwisko, czy_zyje)
VALUES
    ('Zack', 'Little', 1),
    ('Vernice', 'Lowe', 1),
    ('Maeve', 'Stark', 1),
    ('Madisen', 'Abshire', 0),
    ('Ramona', 'Hessel', 1),
    ('Katharina', 'Schinner', 1),
    ('Felipe', 'Collier', 0),
    ('Felicita', 'Hegmann', 1),
    ('Delaney', 'Daugherty', 1),
    ('Kavon', 'Nolan', 0),
    ('Giles', 'Purdy', 1),
    ('Jamar', 'Donnelly', 1),
    ('Jannie', 'Hayes', 0),
    ('Mittie', 'Herzog', 0),
    ('Kyra', 'Becker', 0),
    ('Ernie', 'Emmerich', 1),
    ('Lane', 'Muller', 1),
    ('Lorenza', 'Reichert', 0),
    ('Gilda', 'Hauck', 1),
    ('Russell', 'Pacocha', 1),
    ('Leopoldo', 'Klocko', 1),
    ('Domenico', 'O''Reilly', 1),
    ('Leland', 'Ernser', 1),
    ('Cooper', 'Hickle', 0),
    ('Shaylee', 'Hansen', 0),
    ('Chester', 'Yundt', 1),
    ('Juliet', 'Ullrich', 0),
    ('Frances', 'Marquardt', 0),
    ('Erna', 'Witting-Rodriguez', 1),
    ('Carmine', 'Gorczany', 1),
    ('Rafael', 'Grant', 1),
    ('Darrick', 'Pfeffer', 0),
    ('Rene', 'Weissnat', 0),
    ('Marley', 'Cummerata', 1),
    ('Aliya', 'Huel', 1),
    ('Lavinia', 'Gerlach', 0),
    ('Jacklyn', 'Botsford', 0),
    ('Cayla', 'Bayer', 1),
    ('Tad', 'Anderson', 0),
    ('Everette', 'Schmitt', 1);


INSERT INTO Rada_piekielna
    (kadencja_od, kadencja_do)
VALUES
    ('0001-01-01', NULL)
-- Zakładam, że obecna rada jest pierwszą radą 

INSERT INTO Grzesznik
    (imie, nazwisko, data_smierci, data_urodzenia, status)
VALUES
    (N'Marlon', N'DuBuque', '2000-12-23', '1955-04-10', N'Cierpiący'),
    (N'Pietro', N'Homenick', '2005-02-27', '1990-01-15', N'Cierpiący'),
    (N'Andrew', N'White', '2005-10-15', '1945-11-20', N'Cierpiący'),
    (N'Randi', N'Oberbrunner', '2000-03-15', '1980-08-05', N'Awansowany'),
    (N'Paul', N'Barrows', '2000-10-10', '1962-02-28', N'Cierpiący'),
    (N'Amelie', N'McKenzie', '2000-09-08', '2001-05-14', N'Cierpiący'),
    (N'Sylvia', N'Blick', '2000-12-04', '1938-09-01', N'Cierpiący'),
    (N'Lorine', N'Smith', '2015-05-05', '1975-12-12', N'Cierpiący'),
    (N'Carolina', N'Moen', '2015-04-01', '1950-03-30', N'Cierpiący'),
    (N'Heidi', N'Hoppe', '2015-06-30', '1995-07-07', N'Cierpiący'),
    (N'Euna', N'Jaskolski', '2015-10-27', '1940-10-10', N'Cierpiący'),
    (N'Rozella', N'Nitzsche', '2015-03-23', '1988-01-25', N'Cierpiący'),
    (N'Rex', N'Stehr', '2015-04-19', '1960-06-18', N'Awansowany'),
    (N'Rebeka', N'Bosco', '2015-04-24', '2003-11-02', N'Cierpiący'),
    (N'Florian', N'Padberg', '2015-04-12', '1958-09-19', N'Cierpiący'),
    (N'Kavon', N'Wiegand', '2015-04-30', '1972-02-14', N'Cierpiący'),
    (N'Shanel', N'Kohler', '2015-11-28', '1948-05-30', N'Cierpiący'),
    (N'Abigayle', N'Wiza', '2015-10-17', '1992-08-22', N'Cierpiący'),
    (N'Lura', N'Quitzon', '2015-09-30', '1965-01-05', N'Cierpiący'),
    (N'Harvey', N'Morar', '2015-03-27', '1952-12-12', N'Cierpiący'),
    (N'Eleanore', N'Braun', '2015-04-14', '1985-07-20', N'Cierpiący'),
    (N'Grady', N'Haley', '2015-03-07', '1942-03-15', N'Awansowany'),
    (N'Bulah', N'Altenwerth', '2015-06-14', '1998-10-08', N'Cierpiący'),
    (N'Melody', N'Kovacek-Wintheiser', '2015-06-14', '1970-04-25', N'Cierpiący'),
    (N'Clay', N'Hirthe', '2015-12-06', '1955-09-11', N'Cierpiący'),
    (N'Elmo', N'Considine', '2015-12-19', '1982-11-30', N'Cierpiący'),
    (N'Victoria', N'Nikolaus', '2015-10-08', '1947-02-18', N'Cierpiący'),
    (N'Lindsay', N'Vandervort', '2015-08-10', '1991-06-05', N'Cierpiący'),
    (N'Ernestine', N'Howe', '2015-03-01', '1968-08-28', N'Cierpiący'),
    (N'Owen', N'Adams', '2015-10-24', '1951-01-09', N'Cierpiący'),
    (N'Kevin', N'Wisoky', '2015-05-12', '2000-03-15', N'Cierpiący'),
    (N'Jon', N'Mante', '2015-05-11', '1978-10-22', N'Awansowany'),
    (N'Omari', N'Feest', '2015-07-31', '1949-12-01', N'Cierpiący'),
    (N'Felipe', N'Tromp', '2015-09-29', '1986-05-20', N'Cierpiący'),
    (N'Lenore', N'Dietrich', '2015-11-29', '1961-07-14', N'Cierpiący'),
    (N'Brown', N'Boyer', '2015-11-12', '1996-09-09', N'Cierpiący'),
    (N'Yasmine', N'Mitchell', '2015-03-05', '1953-02-25', N'Cierpiący'),
    (N'Juston', N'Stracke', '2015-08-19', '1974-06-16', N'Cierpiący'),
    (N'Brain', N'Pagac', '2015-05-04', '1944-11-04', N'Cierpiący'),
    (N'Gisselle', N'Abernathy', '2015-03-24', '1993-01-30', N'Cierpiący'),
    (N'Fritz', N'Zulauf', '2015-08-05', '1967-08-12', N'Awansowany'),
    (N'Antoinette', N'Rice', '2015-05-24', '1956-12-24', N'Cierpiący'),
    (N'Krista', N'Nitzsche', '2015-08-29', '1989-03-10', N'Cierpiący'),
    (N'Scot', N'Gleason-Dicki', '2015-07-06', '1946-05-05', N'Cierpiący'),
    (N'Dannie', N'Bergnaum', '2015-01-16', '1973-09-18', N'Cierpiący'),
    (N'Jazlyn', N'Mante', '2015-07-10', '1999-02-14', N'Cierpiący'),
    (N'Dorothea', N'Halvorson', '2015-05-26', '1963-06-20', N'Cierpiący'),
    (N'Elsa', N'Sauer', '2015-07-14', '1984-11-11', N'Cierpiący'),
    (N'Estel', N'Kessler', '2015-06-07', '1954-01-23', N'Cierpiący'),
    (N'Marianne', N'Rolfson', '2015-02-28', '1997-04-09', N'Cierpiący'),
    (N'Easter', N'Kulas', '2015-11-13', '1971-08-30', N'Cierpiący'),
    (N'Monroe', N'O''Conner', '2015-10-23', '1943-12-15', N'Cierpiący'),
    (N'Hubert', N'Hettinger', '2015-05-15', '1987-03-22', N'Cierpiący'),
    (N'Monique', N'Hettinger', '2015-06-15', '1966-07-08', N'Awansowany'),
    (N'Clovis', N'Kuphal', '2015-11-27', '1959-10-01', N'Cierpiący'),
    (N'Jamie', N'Walsh', '2015-01-13', '1994-05-18', N'Cierpiący'),
    (N'Kody', N'Thiel', '2015-06-21', '1977-09-25', N'Cierpiący'),
    (N'Renee', N'Hermiston', '2015-05-19', '1957-01-12', N'Cierpiący'),
    (N'Adelia', N'Lowe', '2015-12-01', '2002-04-16', N'Cierpiący'),
    (N'Aleen', N'Stokes', '2015-06-06', '1981-11-19', N'Cierpiący'),
    (N'Makenna', N'Bernier', '2015-01-19', '1941-06-02', N'Cierpiący'),
    (N'Kobe', N'Romaguera', '2015-03-04', '1995-12-05', N'Cierpiący'),
    (N'Jayne', N'Kuphal-Carter', '2015-08-06', '1969-02-20', N'Cierpiący'),
    (N'Luciano', N'Spinka', '2015-06-13', '1983-08-14', N'Cierpiący'),
    (N'Granville', N'Walsh', '2015-07-05', '1955-10-27', N'Awansowany'),
    (N'Lamont', N'Dietrich', '2015-01-31', '2004-03-08', N'Cierpiący'),
    (N'Leonora', N'Monahan', '2015-05-23', '1976-05-21', N'Cierpiący'),
    (N'Connie', N'Yundt', '2015-11-21', '1948-09-09', N'Cierpiący'),
    (N'Kameron', N'Jacobi', '2015-11-21', '1990-01-01', N'Cierpiący'),
    (N'Ayden', N'Stracke', '2015-10-07', '1964-04-04', N'Cierpiący'),
    (N'Braden', N'Schiller', '2015-11-13', '1988-12-12', N'Cierpiący'),
    (N'Odie', N'Koss', '2015-08-28', '1952-06-28', N'Cierpiący'),
    (N'Colt', N'Bernier', '2015-11-04', '1998-09-17', N'Cierpiący'),
    (N'Gerald', N'Carroll', '2015-04-21', '1970-02-02', N'Cierpiący'),
    (N'Camila', N'Konopelski', '2015-03-11', '1955-07-23', N'Cierpiący'),
    (N'Chanelle', N'Beatty', '2015-03-01', '1982-10-10', N'Awansowany'),
    (N'Nicholaus', N'Conn', '2015-11-07', '1960-01-19', N'Cierpiący'),
    (N'Roxanne', N'Murray', '2015-08-11', '1991-05-25', N'Cierpiący'),
    (N'Esther', N'Bednar', '2015-04-16', '1973-11-03', N'Cierpiący'),
    (N'Gina', N'Yundt', '2015-10-05', '1947-08-18', N'Cierpiący'),
    (N'Nestor', N'Frami', '2015-06-02', '2001-12-30', N'Cierpiący'),
    (N'Anabel', N'Miller', '2015-07-07', '1985-04-11', N'Cierpiący'),
    (N'Alford', N'Rempel', '2015-10-21', '1962-09-06', N'Cierpiący'),
    (N'Aliza', N'Grant', '2015-11-08', '1994-01-28', N'Cierpiący'),
    (N'Brooklyn', N'O''Connell', '2005-07-25', '1958-03-14', N'Cierpiący'),
    (N'Jayson', N'Hettinger', '2005-06-15', '1979-06-29', N'Cierpiący'),
    (N'Doug', N'Shanahan', '2005-09-01', '1944-10-03', N'Cierpiący'),
    (N'Cali', N'Swaniawski', '2005-10-29', '1997-07-17', N'Cierpiący'),
    (N'Vickie', N'Murray', '2005-03-01', '1968-12-08', N'Cierpiący'),
    (N'Ebba', N'Stanton', '2005-09-23', '1983-02-22', N'Cierpiący'),
    (N'Jeremie', N'Hermann', '2005-08-22', '1950-05-13', N'Cierpiący'),
    (N'Jaylin', N'Cartwright-Ward', '2005-11-28', '1992-09-27', N'Awansowany'),
    (N'Zane', N'Gutkowski-Schuster', '2005-02-12', '1975-04-06', N'Cierpiący'),
    (N'Cory', N'Rutherford', '2005-04-30', '1961-01-31', N'Cierpiący'),
    (N'Yasmeen', N'Schinner', '2005-07-30', '1999-08-24', N'Cierpiący'),
    (N'Jaydon', N'Towne', '2005-12-21', '1980-03-03', N'Cierpiący'),
    (N'Justyn', N'Lind-Kozey', '2005-09-04', '1956-11-15', N'Cierpiący'),
    (N'Diego', N'Reichel', '2005-07-31', '1987-05-09', N'Cierpiący'),
    (N'Donato', N'Metz', '2015-07-27', '1965-10-26', N'Cierpiący'),
    (N'Annette', N'Prosacco', '2010-07-27', '1993-06-04', N'Cierpiący');

-- DO TABELI DIABEŁ DODAJE TYLKO CZŁONKÓW RADY I ZWYKŁE DIABŁY, PONIEWAŻ ZGODNIE Z LOGIKĄ PROJEKTU TRWA PIERWSZA RADA, WIĘC NIE ISTNIEJĄ JESZCZE DORADCY
-- Nie dodaję wszystkich 1332 członków rady, na cele projektowe przyjmuję, że pierwsza rada została powołana przez Lucyfera w dużo mniejszym składzie, a dopiero od teraz będzie on powoływał po 1332 członków
INSERT INTO Diabel
    (nazwa, data_powolania, status, diabel_ID_grzesznik)
VALUES
    (N'Mayra', '0001-01-01', N'Członek rady', NULL),
    (N'Blaise', '0001-01-01', N'Członek rady', NULL),
    (N'Lowell', '0001-01-01', N'Członek rady', NULL),
    (N'Alford', '0001-01-01', N'Członek rady', NULL),
    (N'Deonte', '0001-01-01', N'Członek rady', NULL),
    (N'Karlie', '0001-01-01', N'Członek rady', NULL),
    (N'Destinee', '0001-01-01', N'Członek rady', NULL),
    (N'Roslyn', '0001-01-01', N'Członek rady', NULL),
    (N'Dustin', '0001-01-01', N'Członek rady', NULL),
    (N'Una', '0001-01-01', N'Członek rady', NULL),
    (N'Cathrine', '2026-01-11', N'Zwykły', NULL),
    (N'Johnathon', '2025-11-13', N'Zwykły', NULL),
    (N'Samir', '2025-04-28', N'Zwykły', NULL),
    (N'Verla', '2025-02-24', N'Zwykły', NULL),
    (N'Dorthy', '2025-11-30', N'Zwykły', NULL),
    (N'Ivy', '2025-05-10', N'Zwykły', NULL),
    (N'Davonte', '2025-10-26', N'Zwykły', NULL),
    (N'Miles', '2025-12-29', N'Zwykły', NULL),
    (N'Michaela', '2025-01-31', N'Zwykły', NULL),
    (N'Ricardo', '2025-10-20', N'Zwykły', NULL),
    (N'Jeffrey', '2025-03-05', N'Zwykły', NULL),
    (N'Jodie', '2025-04-22', N'Zwykły', NULL),
    (N'Ronny', '2025-11-11', N'Zwykły', NULL),
    (N'Margaret', '2025-06-13', N'Zwykły', NULL),
    (N'Ewald', '2025-11-05', N'Zwykły', NULL),
    (N'Carmella', '2025-02-18', N'Zwykły', NULL),
    (N'Helene', '2025-07-22', N'Zwykły', NULL),
    (N'Edgardo', '2025-10-06', N'Zwykły', NULL),
    (N'Mortimer', '2026-01-10', N'Zwykły', NULL),
    (N'Mohammed', '2025-02-05', N'Zwykły', NULL),
    (N'Buck', '2025-03-22', N'Zwykły', NULL),
    (N'Rashawn', '2025-12-20', N'Zwykły', NULL),
    (N'Conner', '2025-08-23', N'Zwykły', NULL),
    (N'Erling', '2025-03-21', N'Zwykły', NULL),
    (N'Bethany', '2025-02-07', N'Zwykły', NULL),
    (N'Jaleel', '2025-12-10', N'Zwykły', NULL),
    (N'Kristian', '2025-03-05', N'Zwykły', NULL),
    (N'Alexandro', '2025-08-16', N'Zwykły', NULL),
    (N'Elda', '2025-02-28', N'Zwykły', NULL),
    (N'Samanta', '2025-06-16', N'Zwykły', NULL),
    (N'Ernesto', '2025-08-08', N'Zwykły', NULL),
    (N'Griffin', '2025-11-21', N'Zwykły', NULL),
    (N'Janelle', '2025-08-09', N'Zwykły', NULL),
    (N'Isobel', '2025-04-11', N'Zwykły', NULL),
    (N'Bernadette', '2025-02-01', N'Zwykły', NULL),
    (N'Enrique', '2025-04-15', N'Zwykły', NULL);

-- DODANIE DIABŁÓW AWANSOWANYCH Z GRZESZNIKÓW, ŻEBY ZGADZAŁY SIĘ DANE MIĘDZY TABELĄ GRZESZNIK I DIABEŁ
INSERT INTO Diabel
    (nazwa, data_powolania, status, diabel_ID_grzesznik)
SELECT
    nazwisko,
    DATEADD(day, 1, data_smierci),
    N'Zwykły',
    ID_grzesznik
FROM Grzesznik
WHERE status = N'Awansowany';

-- FUNKCJE DIABŁA PRZYDZIELAM DLA DIABŁÓW O STATUSIE 'ZWYKŁY'
INSERT INTO Przydzial_funkcji
    (data_od, data_do, ocena_pracy, przydzial_ID_diabel, przydzial_ID_funkcja)
SELECT
    data_powolania,
    NULL,
    6, -- Na potrzeby zadania ustawiam ocenę wszystkich diabłów na 6 
    ID_diabel,
    CASE 
WHEN ID_diabel % 2 = 0 THEN 2 -- Dla uproszczenia podczas dodawania ustalam, że każdy parzysty to dręczyciel, a nieparzysty to kusiciel
ELSE 1
END
FROM Diabel
WHERE status = N'Zwykły';
-- Bo jeśli ktoś jest członkiem rady albo doradcą, to już nie 'pracuje' jak przeciętny diabeł

-- DODANIE CZŁONKÓW RADY NA PODSTAWIE STATUSU DODANYCH WCZEŚNIEJ DIABŁÓW
INSERT INTO Czlonek_rady
    (powod_wyboru, czlonek_ID_rada, czlonek_ID_diabel)
SELECT
    N'Najwięcej nadzorowanych kar', -- Uproszczenie
    (SELECT TOP 1
        ID_rada
    FROM Rada_piekielna
    WHERE kadencja_do IS NULL),
    ID_diabel
FROM Diabel
WHERE status IN (N'Członek rady');

-- NIE DODAJĘ DORADCÓW LUCYFERA, PONIEWAŻ SKORO DOPIERO TRWA PIERWSZA RADA TO NIE MOGĄ ONI ISTNIEĆ

-- DO ZAPEŁNIENIA WYKAZU GRZECHÓW UŻYWAM ILOCZYNU KARTEZJAŃSKIEGO

INSERT INTO Wpis_grzech
    (wpis_ID_grzesznik, wpis_ID_grzech, ocena_grzechu, data_popelnienia)
SELECT
    gk.ID_grzesznik,
    g.ID_grzech,
    g.skala_max, -- Na potrzeby zadania dodaję tylko grzechy, które zostały ocenione na maksimum w swojej kategorii
    DATEADD(day, -ABS(CHECKSUM(NEWID())) % 3650, gk.data_smierci) -- Grzechy sprzed okolo 10 lat przed smiercia
FROM Grzesznik gk
CROSS JOIN Grzech g-- Losuję 30% możliwych kombinacji
CROSS APPLY (SELECT ABS(CHECKSUM(NEWID())) % 100 AS los) losowanie
WHERE losowanie.los < 30;

-- DODANIE DANYCH DO TABELI WYMIERZONA KARA - NA PODSTAWIE OBLICZEŃ

INSERT INTO Wymierzona_kara
    (
    srednia_ocena_grzechow,
    czestotliwosc_grzechow,
    surowosc,
    data_od,
    data_do,
    kara_ID_kara,
    kara_ID_grzesznik,
    kara_ID_rada
    )
SELECT
    CAST(AVG(w.ocena_grzechu) AS DECIMAL(10,2)),
    CAST(COUNT(*) AS DECIMAL(10,2)) / NULLIF(DATEDIFF(YEAR, g.data_urodzenia, g.data_smierci), 0), -- Częstotliwość liczona jako liczbe grzechów podzieloną przez długość życia
    CASE 
WHEN SUM(w.ocena_grzechu) / 6.0 > 1000 THEN 1000.00 -- Surowość w skali 0-1000
ELSE CAST(SUM(w.ocena_grzechu) / 6.0 AS DECIMAL(10,2))
END,
    DATEADD(day, 1, g.data_smierci),
    CASE
WHEN SUM(w.ocena_grzechu) > 6666 THEN NULL
ELSE DATEADD(YEAR, CAST(SUM(w.ocena_grzechu) / 6.0 AS INT), DATEADD(day, 1, g.data_smierci))
    END,
    (SELECT TOP 1
        ID_kara
    FROM Kara
    ORDER BY NEWID()), -- Losowe ID kary
    g.ID_grzesznik,
    (
SELECT TOP 1
        ID_rada
    FROM Rada_piekielna
    WHERE DATEADD(day, 1, g.data_smierci) >= kadencja_od
        AND (kadencja_do IS NULL OR DATEADD(day, 1, g.data_smierci) <= kadencja_do)
    ORDER BY kadencja_od DESC
    )
FROM Grzesznik g
    JOIN Wpis_grzech w ON w.wpis_ID_grzesznik = g.ID_grzesznik
GROUP BY 
    g.ID_grzesznik, g.data_urodzenia, g.data_smierci;


-- UZUPEŁNIANIE TABELI NADZÓR KARY, NA POTRZEBY ZADANIA USTAWIAM NA RAZIE, ŻE KAŻDA KARA MA TYLKO JEDNEGO NADZORCĘ
-- UŻYWAM TABEL TYMCZASOWYCH, ABY SZTUCZNIE NUMEROWAĆ WIERSZE W NADZÓR KARY I W DIABŁACH, ŻEBY JE POTEM POŁĄCZYĆ ZE SOBĄ

WITH
    dostepni_dreczyciele
    AS
    (
        SELECT
            d.ID_diabel,
            ROW_NUMBER() OVER (ORDER BY NEWID()) AS nr_diabel,
            COUNT(*) OVER () AS ilosc_dreczycieli
        FROM Diabel d
            JOIN Przydzial_funkcji pf ON d.ID_diabel = pf.przydzial_ID_diabel
            JOIN Funkcja_diabla fd ON pf.przydzial_ID_funkcja = fd.ID_funkcja
        WHERE fd.nazwa = N'Dręczyciel'
    ),

    ponumerowane_kary
    AS
    (
        SELECT
            ID_wymierzona_kara,
            data_od,
            data_do,
            ROW_NUMBER() OVER (ORDER BY NEWID()) AS nr_wiersza
        FROM Wymierzona_kara
    )

INSERT INTO Nadzor_kary
    (data_od, data_do, nadzor_ID_diabel, nadzor_ID_kara)
SELECT
    k.data_od,
    k.data_do,
    d.ID_diabel,
    k.ID_wymierzona_kara
FROM
    ponumerowane_kary k
    JOIN dostepni_dreczyciele d ON d.nr_diabel = ((k.nr_wiersza - 1) % d.ilosc_dreczycieli) + 1;

-- DODANIE DANYCH DO TABELI MODLITWY, TAK JAK W PRZYPADKU TABELI GRZECHY KORZYSTAM Z ILOCZYNU KARTEZJAŃSKIEGO 
-- W DANYCH TESTOWYCH MODLITWY NIE WPŁYWAJĄ NA KARY, SĄ JEDYNIE REJESTROWANE W SYSTEMIE, MUSIAŁABYM UŻYĆ PROCEDURY, ABY ZAREJESTROWAĆ ICH WPŁYW NA KARĘ 
INSERT INTO Modlitwa
    (data, modlitwa_ID_osobaZyjaca, modlitwa_ID_grzesznik)
SELECT
    DATEADD(day, ABS(CHECKSUM(NEWID()) % 1096), g.data_smierci), -- Data modlitwy w ciągu 6 lat po śmierci grzesznika
    oz.ID_osoba,
    g.ID_grzesznik
FROM Osoba_zyjaca oz
CROSS JOIN Grzesznik g
CROSS APPLY (SELECT ABS(CHECKSUM(NEWID())) % 100 AS los) losowanie
WHERE oz.czy_zyje = 1
    AND losowanie.los < 10;

-- DODANIE DANYCH DO TABELI KUSZENIE, ANALOGICZNIE JAK W TABELI NADZÓR KARY, W NASZYM PRZYPADKU DODAJĘ DANE KUSZENIA TYLKO OSÓB, KTÓRE MAJĄ ZAZNACZONE ŻE ŻYJĄ, WIEC NP. Z OSTATNICH 6 LAT, UWZGLĘDNIENIE OSÓB, KTÓRE ZMARŁY STANOWIŁOBY UTRUDNIENIE W DOPASOWANIU DIABŁA
WITH
    dostepni_kusiciele
    AS
    (
        SELECT d.ID_diabel
        FROM Diabel d
            JOIN Przydzial_funkcji pf ON d.ID_diabel = pf.przydzial_ID_diabel
            JOIN Funkcja_diabla fd ON pf.przydzial_ID_funkcja = fd.ID_funkcja
        WHERE fd.nazwa = N'Kusiciel'
    )
INSERT INTO Kuszenie
    (data, skutecznosc, kuszenie_ID_grzech, kuszenie_ID_diabel, kuszenie_ID_osoba)
SELECT
    DATEADD(day, -ABS(CHECKSUM(NEWID()) % 2190), GETDATE()), -- Losowa data kuszenia, zakładam ostatnie 6 lat wstecz
    ABS(CHECKSUM(NEWID())) % 2,
    g.ID_grzech, -- Losowy grzech
    k.ID_diabel, -- Losowy diabeł kusiciel
    oz.ID_osoba -- Losowa osoba
FROM Osoba_zyjaca oz
CROSS JOIN dostepni_kusiciele k
CROSS JOIN Grzech g
CROSS APPLY (SELECT ABS(CHECKSUM(NEWID())) % 100 AS los) losowanie
WHERE oz.czy_zyje = 1
    AND losowanie.los < 5;


-- DO TABELI RAPORT WSTAWIAM TYLKO RAPORT Z 'TERAZ', KOLEJNE RAPORTY MOGĄ BYĆ GENEROWANE PROCEDURĄ

INSERT INTO Raport
    (
    data,
    liczba_diablow_lacznie,
    liczba_diablow_zwyklych,
    liczba_czlonkow_rady,
    liczba_doradcow,
    liczba_grzesznikow,
    zmiana_diablow,
    zmiana_grzesznikow,
    raport_ID_rada
    )
SELECT
    GETDATE(),
    (SELECT COUNT(*)
    FROM Diabel),
    (SELECT COUNT(*)
    FROM Diabel
    WHERE status = N'Zwykły'),
    (SELECT COUNT(*)
    FROM Diabel
    WHERE status = N'Członek rady'),
    (SELECT COUNT(*)
    FROM Diabel
    WHERE status = N'Doradca Lucyfera'),
    (SELECT COUNT(*)
    FROM Wymierzona_kara
    WHERE data_do IS NULL OR data_do >= GETDATE()), -- Wybieramy tylko grzeszników, których kara trwa, bo jeśli zakończyła się, to mogą teoretycznie opuścić piekło
    0, -- Pierwszy raport, brak porównania z wcześniejszymi
    0,
    (SELECT TOP 1
        ID_rada
    FROM Rada_piekielna
    WHERE GETDATE() >= kadencja_od
        AND (kadencja_do IS NULL OR GETDATE() <=kadencja_do)
    ORDER BY kadencja_od DESC
    );

(40 rows affected)

(1 row affected)

(100 rows affected)

(46 rows affected)

(9 rows affected)

...

(100 rows affected)

(235 rows affected)

(440 rows affected)

(1 row affected)

Total execution time: 00:00:01.086

## 3. Realizacja zapytań i procedur w języku SQL

**Zapytania**


Zapytanie nr 1 dotyczy osób żyjących, które najwięcej modlą się za grzeszników. Umożliwia ono wyznaczenie celów do kuszenia, ponieważ modlitwy niosą zagrożenie stabilności kar w piekle.

In [119]:
SELECT
    oz.ID_osoba,
    oz.imie,
    oz.nazwisko,
    COUNT(m.ID_modlitwa) AS liczba_modlitw,
    MIN(m.data) AS pierwsza_modlitwa,
    MAX(m.data) AS ostatnia_modlitwa -- Bo może okazać się, że ktoś nie modlił się np. od 20 lat, ale wcześniej modlił się codziennie, więc dalej jest w gronie najgorliwszych
FROM Modlitwa m
    JOIN Osoba_zyjaca oz ON m.modlitwa_ID_osobaZyjaca = oz.ID_osoba
GROUP BY oz.ID_osoba, oz.imie, oz.nazwisko
ORDER BY liczba_modlitw DESC;

(24 rows affected)

Total execution time: 00:00:00.078

ID_osoba,imie,nazwisko,liczba_modlitw,pierwsza_modlitwa,ostatnia_modlitwa
20,Russell,Pacocha,16,2002-11-08,2017-11-21
11,Giles,Purdy,15,2003-08-31,2018-03-05
16,Ernie,Emmerich,14,2005-08-18,2018-06-30
34,Marley,Cummerata,14,2005-11-05,2018-10-18
26,Chester,Yundt,12,2008-06-04,2018-07-24
8,Felicita,Hegmann,12,2006-07-15,2018-11-23
5,Ramona,Hessel,12,2002-03-04,2018-03-23
29,Erna,Witting-Rodriguez,11,2007-02-23,2018-04-12
17,Lane,Muller,11,2001-03-13,2018-10-05
38,Cayla,Bayer,11,2005-02-20,2018-02-04


Zapytanie nr 2 dotyczy analogicznie grzeszników, za których najczęściej się modlono, czyli takich, dla których istnieje realne zagrożenie wygaśnięciem kary. Co ważne, w tej sekcji wypisujemy również datę zakończenia kary, ponieważ zgodnie z procedurą utworzoną w następnych krokach dla wpływu modlitw na karę, jeśli czyjaś kara ma trwać nieskończoność, to modliwy nie będą wpływać na długość kary, a jedynie zmieniać surowość. W takim wypadku zagrożenie jest mniejsze, ale nadal warto monitorować stan surowości kary.

In [120]:
SELECT
    g.ID_grzesznik,
    g.imie,
    g.nazwisko,
    COUNT(m.ID_modlitwa) AS liczba_modlitw,
    MAX(m.data) AS data_ostatniej_modlitwy,
    wk.surowosc,
    wk.data_do
FROM Modlitwa m
    JOIN Grzesznik g ON m.modlitwa_ID_grzesznik = g.ID_grzesznik
    JOIN Wymierzona_kara wk ON g.ID_grzesznik = wk.kara_ID_grzesznik
GROUP BY 
    g.ID_grzesznik, g.imie, g.nazwisko, wk.surowosc, wk.data_do
ORDER BY 
    liczba_modlitw DESC;

(89 rows affected)

Total execution time: 00:00:00.047

ID_grzesznik,imie,nazwisko,liczba_modlitw,data_ostatniej_modlitwy,surowosc,data_do
73,Colt,Bernier,7,2018-05-14,45.00,2060-11-05
59,Adelia,Lowe,6,2018-04-19,61.67,2076-12-02
93,Zane,Gutkowski-Schuster,6,2007-04-26,20.00,2025-02-13
99,Donato,Metz,5,2017-12-08,23.33,2038-07-28
61,Makenna,Bernier,5,2017-05-31,45.00,2060-01-20
42,Antoinette,Rice,5,2017-12-18,15.00,2030-05-25
43,Krista,Nitzsche,5,2017-05-19,25.00,2040-08-30
80,Gina,Yundt,5,2018-06-10,40.00,2055-10-06
63,Jayne,Kuphal-Carter,5,2018-03-23,48.33,2063-08-07
36,Brown,Boyer,5,2018-10-06,41.67,2056-11-13


Zapytanie nr 3 służy do wyznaczania najmniej obciążonych dręczeniem diabłów, w przypadku, gdy liczba wszystkich diabłów w piekle przekracza 100 000 000. Obciążenie diabła jest liczone na podstawie aktywnych nadzorów kar, które sprawuje, a do analizy wybierani są wyłącznie zwykłe diabły o aktywnej funkcji dręczyciela. Dzięki temu system może wytypować najmniej zajęte diabły do kuszenia żywych. W moich danych testowych nie ma nadwyżki, więc zapytanie nic nie zwróci.

In [121]:
-- Do utworzenia tej tabeli używam dwoch tabeli tymczasowych, pierwsza sprawdza ilość diabłów w piekle, a druga pokazuje tylko diabły z aktualnymi funkcjami
WITH
    LiczbaDiablow
    AS
    (
        SELECT COUNT(*) AS liczba_diablow
        FROM Diabel
    ),

    AktualneFunkcje
    AS
    (
        SELECT
            pf.przydzial_ID_diabel AS ID_diabel,
            fd.nazwa
        FROM Przydzial_funkcji pf
            JOIN Funkcja_diabla fd ON pf.przydzial_ID_funkcja = fd.ID_funkcja
        WHERE pf.data_do is NULL
    )

SELECT
    d.ID_diabel,
    d.nazwa,
    COUNT(nk.ID_nadzor) AS liczba_nadzorowanych_kar
FROM Diabel d
    JOIN LiczbaDiablow ld ON 1=1
    JOIN AktualneFunkcje af ON d.ID_diabel = af.ID_diabel
    LEFT JOIN Nadzor_kary nk ON d.ID_diabel = nk.nadzor_ID_diabel
        AND (nk.data_do IS NULL OR nk.data_do >= GETDATE())
WHERE ld.liczba_diablow > 100000000
    AND d.status = N'Zwykły'
    AND af.nazwa = N'Dręczyciel'
GROUP BY d.ID_diabel, d.nazwa
ORDER BY liczba_nadzorowanych_kar ASC;

(0 rows affected)

Total execution time: 00:00:00.040

ID_diabel,nazwa,liczba_nadzorowanych_kar


Zapytanie nr 4 dotyczy wytypowania diabłów, którzy mają na swoim koncie najwięcej sukcesów - zarówno w zakresie dręczenia grzeszników, jak i kuszenia osób żyjących. Za jego pomocą można sprawdzić, które diabły są w danym momencie kandydatami do rady piekielnej. Zapytanie zostało podzielone na cztery podzapytania, które służą wyświetlaniu po 333 diabłów z największymi sukcesami w zakresie:

A. kuszenia (tych, którzy skusili najwięcej osób)

In [122]:
SELECT TOP 333
    d.ID_diabel,
    d.nazwa,
    COUNT(*) AS liczba_kuszen
FROM Kuszenie k
    JOIN Diabel d ON k.kuszenie_ID_diabel = d.ID_diabel
    JOIN Osoba_zyjaca oz ON k.kuszenie_ID_osoba = oz.ID_osoba
WHERE oz.czy_zyje = 1
    AND k.skutecznosc = 1
    AND d.status = N'Zwykły'
GROUP BY d.ID_diabel, d.nazwa
ORDER BY liczba_kuszen DESC;

(25 rows affected)

Total execution time: 00:00:00.048

ID_diabel,nazwa,liczba_kuszen
53,Stehr,15
59,Beatty,12
1,Belzebub,12
29,Margaret,12
35,Mohammed,12
31,Carmella,11
43,Alexandro,11
17,Johnathon,11
25,Ricardo,11
39,Erling,10


B. kuszenia gorliwych (tych, którzy skusili najwięcej modlących się osób) - w przypadku moich danych testowych zarówno zapytanie o TOP kusicieli i TOP kusicieli gorliwych zwraca ten sam wynik, bo mam na tyle mało osób żyjących, że wszystkie zaliczają się jako gorliwe

In [123]:
-- Najpierw wyznaczam najbardziej gorliwe osoby żyjące
WITH
    Gorliwi
    AS
    (
        SELECT TOP 100
        modlitwa_ID_osobaZyjaca
        FROM Modlitwa
        GROUP BY modlitwa_ID_osobaZyjaca
        ORDER BY COUNT(*) DESC
    )

SELECT TOP 333
    d.ID_diabel,
    d.nazwa,
    COUNT(*) AS liczba_kuszen
FROM Kuszenie k
    JOIN Diabel d ON k.kuszenie_ID_diabel = d.ID_diabel
    JOIN Gorliwi g ON k.kuszenie_ID_osoba = g.modlitwa_ID_osobaZyjaca
WHERE k.skutecznosc = 1
    AND d.status = N'Zwykły'
GROUP BY d.ID_diabel, d.nazwa
ORDER BY liczba_kuszen DESC;

(25 rows affected)

Total execution time: 00:00:00.047

ID_diabel,nazwa,liczba_kuszen
53,Stehr,15
59,Beatty,12
1,Belzebub,12
29,Margaret,12
35,Mohammed,12
31,Carmella,11
43,Alexandro,11
17,Johnathon,11
25,Ricardo,11
39,Erling,10


C. kuszenia na najcięższe grzechy

In [124]:
SELECT TOP 333
    d.ID_diabel,
    d.nazwa,
    COUNT(*) AS ciezkie_kuszenia
FROM Kuszenie k
    JOIN Diabel d ON k.kuszenie_ID_diabel = d.ID_diabel
    JOIN Grzech g ON k.kuszenie_ID_grzech = g.ID_grzech
WHERE g.kategoria = N'Ciężki'
    AND k.skutecznosc = 1
    AND d.status = N'Zwykły'
GROUP BY d.ID_diabel, d.nazwa
ORDER BY ciezkie_kuszenia DESC;

(22 rows affected)

Total execution time: 00:00:00.043

ID_diabel,nazwa,ciezkie_kuszenia
1,Belzebub,7
23,Miles,5
35,Mohammed,5
47,Griffin,5
59,Beatty,5
49,Isobel,4
31,Carmella,4
33,Edgardo,4
39,Erling,4
17,Johnathon,4


D. liczby nadzorowanych kar - w tym zapytaniu zakładam, że chodzi o diabły, które mają największą ilość nadzorów zarówno aktywnych, jak i zakończonych, bo liczy się doświadczenie, a nie aktualne obłożenie - dla mojej bazy danych, przydzielałam nadzory kar pomiędzy dostępne diabły równomiernie, dlatego większość sprawuje po 4 nadzory

In [125]:
SELECT TOP 333
    d.ID_diabel,
    d.nazwa,
    COUNT(*) AS liczba_nadzorow
FROM Nadzor_kary nk
    JOIN Diabel d ON nk.nadzor_ID_diabel = d.ID_diabel
WHERE d.status = N'Zwykły'
GROUP BY d.ID_diabel, d.nazwa
ORDER BY liczba_nadzorow DESC;

(25 rows affected)

Total execution time: 00:00:00.050

ID_diabel,nazwa,liczba_nadzorow
2,Asmodeusz,4
4,Mefistofeles,4
16,Cathrine,4
18,Samir,4
20,Dorthy,4
22,Davonte,4
24,Michaela,4
26,Jeffrey,4
28,Ronny,4
30,Ewald,4


Zapytanie nr 5 dotyczy generowania aktualnych statystyk piekła, na wypadek, gdyby Lucyfer chciał sprawdzić stan piekła 'na teraz', bez porównywania danych z poprzednimi okresami. Zapisanie tego zapytania przyda się później do stworzenia procedury tworzenia raportu.

In [126]:
SELECT
    (SELECT COUNT(*)
    FROM Diabel) AS liczba_diablow_lacznie,

    (SELECT COUNT(*)
    FROM Diabel
    WHERE status = N'Zwykły') AS liczba_diablow_zwyklych,

    (SELECT COUNT(*)
    FROM Diabel
    WHERE status = N'Członek rady') AS liczba_czlonkow_rady,

    (SELECT COUNT(*)
    FROM Diabel
    WHERE status = N'Doradca Lucyfera') AS liczba_doradcow,

    (SELECT COUNT(*)
    FROM Wymierzona_kara
    WHERE data_do IS NULL OR data_do >= GETDATE())
    AS liczba_grzesznikow_w_piekle;

(1 row affected)

Total execution time: 00:00:00.041

liczba_diablow_lacznie,liczba_diablow_zwyklych,liczba_czlonkow_rady,liczba_doradcow,liczba_grzesznikow_w_piekle
60,50,10,0,94


**Procedury**

Bazą systemu piekielnego jest procedura _OsadzGrzesznika_, której celem jest dobranie surowości i długości kary dla danego grzesznika w oparciu o wpisy grzechów. Opis piekła nie wspomina o tym, w jaki sposób wyznaczana jest surowość czy długość kary, dlatego na cele projektowe stworzyłam własne założenia. Surowość jest liczona w skali 0-1000 i określa się ją poprzez zsumowanie ocen wszystkich popełnionych przez grzesznika grzechów i podzieleniu ich przez 6. Jeśli wynik jest wyższy niż 1000, to ustawiona jest maksymalna surowość wynosząca 1000. Długość kary również wynika z sumy grzechów -  jeśli suma grzechów jest większa niż 6666 to kara jest wieczna, jeśli suma jest mniejsza to dodajemy lata w liczbie suma ocen grzechów / 6.

In [111]:
CREATE PROCEDURE OsadzGrzesznika
    @ID_grzesznik INT
AS
BEGIN
    SET NOCOUNT ON;
    -- Wyłączam zwracanie komunikatów o liczbie przetworzonych wierszy

    -- Sprawdzam, czy grzesznik nie ma już wymierzonej kary
    IF EXISTS (SELECT 1
    FROM Wymierzona_kara
    WHERE kara_ID_grzesznik = @ID_grzesznik)
    BEGIN
        RAISERROR('Ten grzesznik został już ukarany.', 16, 1);
        RETURN;
    END;

    INSERT INTO Wymierzona_kara
        (
        srednia_ocena_grzechow,
        czestotliwosc_grzechow,
        surowosc,
        data_od,
        data_do,
        kara_ID_kara,
        kara_ID_grzesznik,
        kara_ID_rada
        )
    SELECT
        CAST(AVG(w.ocena_grzechu) AS DECIMAL(10,2)) AS srednia_ocena,

        CAST(COUNT(*) AS DECIMAL(10,2)) / NULLIF(DATEDIFF(YEAR, g.data_urodzenia, g.data_smierci), 0) AS czestotliwosc, -- NULLIF jako zabezpieczenie przed dzieleniem przez 0

        CASE
            WHEN SUM(w.ocena_grzechu) / 6.0 > 1000 THEN 1000.00 -- Suma ocen grzechów podzielona przez 6 (moje własne założenie) wyznacza surowość kary, max 1000
            ELSE CAST(SUM(w.ocena_grzechu) / 6.0 AS DECIMAL(10,2))
        END AS surowosc_finalna,

        DATEADD(day, 1, g.data_smierci) AS data_od,

        CASE
            WHEN SUM(w.ocena_grzechu) > 6666 THEN NULL -- Jeśli suma ocen grzechów większa niż 6666, to kara będzie wieczna (moje własne założenie)
            ELSE DATEADD(
                    YEAR,
                    CAST(SUM(w.ocena_grzechu) / 6.0 AS INT),
                    DATEADD(day, 1, g.data_smierci)
                 )
        END AS data_do,

        -- Kary przydzielam losowo
        (SELECT TOP 1
            ID_kara
        FROM Kara
        ORDER BY NEWID()) AS losowa_kara,

        g.ID_grzesznik,

        -- Odpowiednią radę dodaję poprzez sprawdzenie, czyja kadencja trwała w dniu śmierci grzesznika + 1 dzień (zakładam, że wtedy jest osądzony)
        (
            SELECT TOP 1
            ID_rada
        FROM Rada_piekielna
        WHERE DATEADD(day, 1, g.data_smierci) >= kadencja_od
            AND (kadencja_do IS NULL OR DATEADD(day, 1, g.data_smierci) <= kadencja_do)
        ORDER BY kadencja_od DESC
        ) AS odpowiednia_rada

    FROM Grzesznik g
        JOIN Wpis_grzech w ON g.ID_grzesznik = w.wpis_ID_grzesznik
    WHERE g.ID_grzesznik = @ID_grzesznik
    GROUP BY g.ID_grzesznik, g.data_urodzenia, g.data_smierci;

END;
GO

Commands completed successfully.

Total execution time: 00:00:00.049

Procedura _PrzydzielNadzor_ odpowiada za przydzielanie diabłów do nadzorowania wymierzonych kar grzeszników. W procedurze zaimplementowano walidację danych wejściowych, w tym sprawdzenie istnienia diabła oraz kary, a także weryfikację, czy diabeł posiada aktualnie funkcję dręczyciela.  
Istotnym elementem logiki biznesowej jest kontrola obciążenia diabła - procedura uniemożliwia przydzielenie nowego nadzoru w sytuacji, gdy diabeł nadzoruje już maksymalną dopuszczalną liczbę 1000 aktywnych kar.

In [112]:
CREATE PROCEDURE PrzydzielNadzor
    @ID_diabel INT,
    @ID_wymierzona_kara INT,
    @data_od DATE NULL
AS
BEGIN
    SET NOCOUNT ON;

    -- Używam zawsze dzisiejszej daty
    IF @data_od IS NULL
    SET @data_od = GETDATE();

    -- Zanim przejdę do właściwej procedury sprawdzam, czy wszystkie wprowadzone podany diabeł i kara istnieją w bazie danych i czy kara nie ma nadzoru, który trwa
    IF NOT EXISTS (
    SELECT 1
    FROM Diabel
    WHERE ID_diabel = @ID_diabel)
    BEGIN
        RAISERROR('Podany diabeł nie istnieje.', 16, 1);
        RETURN;
    END;

    IF NOT EXISTS (
    SELECT 1
    FROM Wymierzona_kara
    WHERE ID_wymierzona_kara = @ID_wymierzona_kara)
    BEGIN
        RAISERROR('Podana kara nie istnieje.', 16, 1);
        RETURN;
    END;

    IF NOT EXISTS (
    SELECT 1
    FROM Przydzial_funkcji pf
        JOIN Funkcja_diabla fd ON pf.przydzial_ID_funkcja = fd.ID_funkcja
    WHERE pf.przydzial_ID_diabel = @ID_diabel
        AND fd.nazwa = N'Dręczyciel'
        AND pf.data_do IS NULL)
    BEGIN
        RAISERROR('Podany diabeł nie ma funkcji dręczenia.', 16, 1);
        RETURN;
    END;

    IF EXISTS (
    SELECT 1
    FROM Nadzor_kary
    WHERE nadzor_ID_kara = @ID_wymierzona_kara
        AND data_do IS NULL)
    BEGIN
        RAISERROR('Ta kara jest aktualnie nadzorowana. Zakończ obecny nadzór kary, aby dodać nowy.', 16, 1);
        RETURN;
    END;

    DECLARE @LiczbaAktywnychKar INT;
    SELECT @LiczbaAktywnychKar = COUNT(*)
    FROM Nadzor_kary nk
    WHERE nk.nadzor_ID_diabel = @ID_diabel
        AND nk.data_do IS NULL;

    IF @LiczbaAktywnychKar >= 1000
    BEGIN
        RAISERROR ('Diabeł nie może nadzorować więcej niż 1000 grzeszników na raz.', 16, 1);
        RETURN;
    END;

    INSERT INTO Nadzor_kary
        (data_od, data_do, nadzor_ID_diabel, nadzor_ID_kara)
    VALUES
        (@data_od, NULL, @ID_diabel, @ID_wymierzona_kara);
END;

Commands completed successfully.

Total execution time: 00:00:00.050

Procedura _DodajModlitwe_ realizuje proces rejestrowania modlitw osób żyjących za grzeszników oraz automatycznie aktualizuje parametry wymierzonej kary. Zgodnie z przyjętymi założeniami projektowymi każda modlitwa powoduje zmniejszenie surowości kary oraz skrócenie jej czasu trwania, przy czym kary wieczne nie podlegają skracaniu.  
Dzięki temu procedura odzwierciedla wpływ modlitw na funkcjonowanie systemu piekielnego oraz zapewnia spójność danych w bazie.

In [113]:
CREATE PROCEDURE DodajModlitwe
    @ID_osoba INT,
    @ID_grzesznik INT,
    @data DATE = NULL
AS
BEGIN
    SET NOCOUNT ON;

    IF @data IS NULL
    SET @data = GETDATE();

    -- Sprawdzam, czy podana osoba jest w spisie osób żyjących i czy na pewno jeszcze żyje
    IF NOT EXISTS (
    SELECT 1
    FROM Osoba_zyjaca
    WHERE ID_osoba = @ID_osoba
        AND czy_zyje = 1)
    BEGIN
        RAISERROR('Ta osoba nie istnieje lub nie żyje.', 16, 1);
        RETURN;
    END;

    -- Sprawdzam, czy grzesznik istnieje
    IF NOT EXISTS (
    SELECT 1
    FROM Grzesznik
    WHERE ID_grzesznik = @ID_grzesznik)
    BEGIN
        RAISERROR('Ten grzesznik nie istnieje.', 16, 1);
        RETURN;
    END;

    -- Sprawdzam, czy grzesznik odbywa karę, na którą może wpłynąć modlitwa
    IF NOT EXISTS (
    SELECT 1
    FROM Wymierzona_kara
    WHERE kara_ID_grzesznik = @ID_grzesznik
        AND (data_do IS NULL OR data_do >= @data))
    BEGIN
        RAISERROR('Ten grzesznik nie ma aktywnej kary.', 16, 1);
        RETURN;
    END;

    -- Wstawiam nową modlitwę do tabeli modlitwa
    INSERT INTO Modlitwa
        (data, modlitwa_ID_osobaZyjaca, modlitwa_ID_grzesznik)
    VALUES
        (@data, @ID_osoba, @ID_grzesznik);

    -- Aktualizuje wymierzoną karę, zgodnie z moimi założeniami każda modlitwa zmniejsza surowość kary o 0.5 (nie może spaść do 0) i długość kary o dzień. Wyjątkiem jest kara wieczna, której długość nie może być zmieniona
    UPDATE Wymierzona_kara
    SET 
    surowosc = CASE
    WHEN surowosc > 0.5 THEN surowosc - 0.5
    ELSE surowosc
    END,
    data_do = CASE
    WHEN data_do IS NOT NULL THEN DATEADD(day, -1, data_do)
    ELSE NULL
    END
    WHERE kara_ID_grzesznik = @ID_grzesznik;
END;
GO

: Msg 2714, Level 16, State 3, Procedure DodajModlitwe, Line 1
There is already an object named 'DodajModlitwe' in the database.

Total execution time: 00:00:00.065

Procedura _ZmienFunkcjeDiabla_ realizuje proces zmiany funkcji diabłów, zgodnie z zasadami obowiązującymi w systemie piekielnym. Zmiana funkcji może nastąpić raz na 66 lat i dotyczy wyłącznie diabłów, których ocena pracy w aktualnym przydziale jest niska. Na potrzeby projektu przyjęto założenie, że do zmiany funkcji kwalifikują się diabły z oceną pracy mniejszą lub równą 3/10. Procedura kończy bieżący przydział funkcji oraz przypisuje diabłu nową funkcję, rozpoczynającą się dzień po zakończeniu poprzedniej, co zapobiega nakładaniu się okresów obowiązywania funkcji.

In [114]:
CREATE PROCEDURE ZmienFunkcjeDiabla
    @ID_diabel INT,
    @data_zmiany DATE = NULL
AS
BEGIN
    SET NOCOUNT ON;

    IF @data_zmiany IS NULL
    SET @data_zmiany = GETDATE();

    -- Zanim przejdę do właściwej procedury sprawdzam, czy diabeł istnieje 
    IF NOT EXISTS (
    SELECT 1
    FROM Diabel
    WHERE ID_diabel = @ID_diabel)
    BEGIN
        RAISERROR('Podany diabeł nie istnieje.', 16, 1);
        RETURN;
    END;

    DECLARE @ID_funkcja INT,
    @data_od_starej DATE,
    @ocena INT;
    SELECT TOP 1
        @ID_funkcja = pf.przydzial_ID_funkcja,
        @data_od_starej = pf.data_od,
        @ocena = pf.ocena_pracy
    FROM Przydzial_funkcji pf
    WHERE pf.przydzial_ID_diabel = @ID_diabel
        AND pf.data_do IS NULL;

    IF @ID_funkcja IS NULL
    BEGIN
        RAISERROR('Ten diabeł nie ma aktywnej funkcji.', 16, 1);
        RETURN;
    END;

    IF (DATEDIFF(YEAR, @data_od_starej, @data_zmiany) < 66 OR @ocena > 3)
    BEGIN
        RAISERROR('Od ostatniej zmiany funkcji diabła nie minęło jeszcze 66 lat lub ocena pracy nie spełnia wymogów zmiany. ', 16, 1);
        RETURN;
    END;

    -- Kończę poprzedni przydział funkcji
    UPDATE Przydzial_funkcji
    SET data_do = @data_zmiany
    WHERE przydzial_ID_diabel = @ID_diabel
        AND data_do IS NULL;

    DECLARE @NowaFunkcja INT;

    SET @NowaFunkcja =
    CASE 
        WHEN @ID_funkcja = 1 THEN 2
        WHEN @ID_funkcja = 2 THEN 1
        ELSE NULL
    END;

    INSERT INTO Przydzial_funkcji
        (data_od, data_do, ocena_pracy, przydzial_ID_diabel, przydzial_ID_funkcja)
    VALUES
        (DATEADD(day, 1, @data_zmiany), NULL, 5, @ID_diabel, @NowaFunkcja); -- Na start zakładam, że każdy diabeł dostaje ocenę 5/10, nowa funkcja ma też data_od jako dzień później od zakończenia poprzedniej funkcji, aby uniknąć zazębiania się dat
END;
GO

Commands completed successfully.

Total execution time: 00:00:00.047

Procedura _NadwyzkaDiablow_ automatyzuje proces przekwalifikowania części dręczycieli na kusicieli w sytuacji zbyt dużego przyrostu populacji diabłów. Jeśli liczba diabłów przekroczy ustalony limit 100 000 000, procedura identyfikuje nadwyżkę oraz wyznacza odpowiednią liczbę najmniej obciążonych dręczycieli, wykorzystując bieżące dane o aktywnych nadzorach kar. Procedura zamyka ich obecne przydziały funkcji oraz przydziela im nową funkcję „Kusiciel”, zachowując ciągłość historii i spójność danych.

In [115]:
CREATE PROCEDURE NadwyzkaDiablow
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @Limit INT = 100000000;
    DECLARE @Nadwyzka INT;

    -- Obliczam nadwyżkę diabłów
    SELECT @Nadwyzka =
    CASE WHEN COUNT(*) > @Limit THEN COUNT(*) - @Limit
    ELSE 0
    END
    FROM Diabel;

    IF @Nadwyzka = 0
    BEGIN
        PRINT 'Brak nadwyżki diabłów.';
        RETURN;
    END;

    DECLARE @FunkcjaKusiciel INT;
    SELECT @FunkcjaKusiciel = ID_funkcja
    FROM Funkcja_diabla
    WHERE nazwa = N'Kusiciel';

    WITH
        NajmniejObciazeni
        AS
        (
            SELECT TOP (@Nadwyzka)
                d.ID_diabel
            FROM Diabel d
                JOIN Przydzial_funkcji pf ON d.ID_diabel = pf.przydzial_ID_diabel AND pf.data_do IS NULL -- AND jest tutaj a nie w WHERE, ponieważ zależy mi, aby liczyć tylko aktywne nadzory, ale nie usuwać diabłów, którzy mają ich 0, bo to oni są głównymi kandydatami do zmiany
                JOIN Funkcja_diabla fd ON pf.przydzial_ID_funkcja = fd.ID_funkcja
                LEFT JOIN Nadzor_kary nk ON d.ID_diabel = nk.nadzor_ID_diabel AND nk.data_do IS NULL
            WHERE d.status = N'Zwykły'
                AND fd.nazwa = N'Dręczyciel'
            GROUP BY d.ID_diabel
            ORDER BY COUNT(nk.ID_nadzor) ASC
        )

    -- Zamknięcie starych przydziałów funkcji
    UPDATE Przydzial_funkcji
    SET data_do = GETDATE()
    WHERE przydzial_ID_diabel IN (SELECT ID_diabel
        FROM NajmniejObciazeni)
        AND data_do IS NULL;

    -- Dodanie nowych przydziałów
    INSERT INTO Przydzial_funkcji
        (data_od, data_do, ocena_pracy, przydzial_ID_diabel, przydzial_ID_funkcja)
    SELECT
        DATEADD(day, 1, GETDATE()),
        NULL,
        5,
        ID_diabel,
        @FunkcjaKusiciel
    FROM NajmniejObciazeni;

    PRINT 'Przekwalifikowano ' + CAST(@Nadwyzka AS VARCHAR) + ' dręczycieli na kusicieli.';
END;
GO

Commands completed successfully.

Total execution time: 00:00:00.049

Procedura _ZakonczRade_ odpowiada za obsługę zakończenia kadencji rady piekielnej, zarówno w przypadku jej wygaśnięcia po 6666 latach, jak i wcześniejszego rozwiązania z inicjatywy niezadowolonego Lucyfera. Procedura znajduje aktualnie działającą radę, sprawdza długość jej kadencji, a następnie wykonuje odpowiednie działania dotyczące członków rady. W przypadku wygaśnięcia kadencji procedura automatycznie awansuje członków rady na stanowiska doradców Lucyfera, nadając im losowo wybraną sferę działania oraz zachowując historię ich aktywności. Jeśli rada zostanie rozwiązana przed upływem kadencji, procedura przywraca wszystkim członkom status zwykłych diabłów.

In [116]:
CREATE PROCEDURE ZakonczRade
    @DataZakonczenia DATE = NULL
AS
BEGIN
    SET NOCOUNT ON;

    IF @DataZakonczenia IS NULL
    SET @DataZakonczenia = GETDATE();

    DECLARE @DataStartu DATE;
    DECLARE @CzyPelnaKadencja BIT;

    SELECT TOP 1
        @DataStartu = kadencja_od
    FROM Rada_piekielna
    WHERE kadencja_do IS NULL;

    IF @DataStartu IS NULL
    BEGIN
        RAISERROR('Brak aktywnej rady do zakończenia.', 16, 1);
        RETURN;
    END;

    -- Sprawdzam, czy minęło 6666 lat
    IF DATEDIFF(YEAR, @DataStartu, @DataZakonczenia) >= 6666
        SET @CzyPelnaKadencja = 1;
    ELSE
        SET @CzyPelnaKadencja = 0;

    -- Jeśli rada ukończyła pełną kadencję, to członkowie rady awansują na stanowisko doraców Lucyfera, dlatego muszę dodać ich dane do tabeli Doradca
    IF @CzyPelnaKadencja = 1
    BEGIN
        INSERT INTO Doradca_lucyfera
            (sfera_dzialania, data_nominacji, doradca_ID_diabel, doradca_ID_czlonek)
        SELECT
            s.sfera,
            DATEADD(DAY, 1, @DataZakonczenia),
            d.ID_diabel,
            cr.ID_czlonek
        FROM Diabel d
            JOIN Czlonek_rady cr ON d.ID_diabel = cr.czlonek_ID_diabel
    
        -- Używam CROSS APPLY, aby stworzyć tasowanie pomiędzy sferami, SELECT UNION ALL tworzy małą tabelę z podanymi sferami, CROSS APPLY je tasuje, a TOP 1 sprawia, że zawsze bierzemy pierwszą sferę z góry
        CROSS APPLY (
            SELECT TOP 1
                sfera
            FROM (
                SELECT 'Ochrona zdrowia' AS sfera
                UNION ALL
                    SELECT 'Bezpieczeństwo obywateli'
                UNION ALL
                    SELECT 'Sprawy wewnętrzne'
                UNION ALL
                    SELECT 'Relacje międzyswiatowe'
            ) opcje -- opcje to nazwa tej tymczasowej tabeli
            ORDER BY NEWID()
        ) s
        WHERE d.status = N'Członek rady';
    END;

    -- Teraz aktualizuję statusy diabłów
    UPDATE Diabel
    SET status =
        CASE
            WHEN @CzyPelnaKadencja = 1 THEN N'Doradca Lucyfera'
            ELSE N'Zwykły'
        END
    WHERE status = N'Członek rady';

    -- Następnie zamykam kadencję rady
    UPDATE Rada_piekielna
    SET kadencja_do = @DataZakonczenia
    WHERE kadencja_do IS NULL;

    PRINT
        CASE
            WHEN @CzyPelnaKadencja = 1
                THEN 'Kadencja rady piekielnej została zakończona. Członkowie zostali doradcami Lucyfera.'
            ELSE
                'Rada piekielna została rozwiązana przed końcem kadencji. Członkowie wrócili do roli zwykłych diabłów.'
        END;
END;
GO

Commands completed successfully.

Total execution time: 00:00:00.048

Procedura _UtworzRade_ odpowiada za automatyczne powołanie nowej rady piekielnej, zgodnie z zasadami przedstawionymi w case study. Po utworzeniu nowej kadencji procedura identyfikuje cztery grupy po 333 najaktywniejszych diabłów, wykorzystując różne kryteria efektywności ich działań: <span data-start="3142" data-end="3174" style="color: var(--vscode-foreground);">najwięcej skutecznych kuszeń,&nbsp;</span>  <span data-start="3181" data-end="3253" style="color: var(--vscode-foreground);">najwięcej kuszeń osób gorliwych,</span> <span data-start="3260" data-end="3314" style="color: var(--vscode-foreground);">najwięcej kuszeń prowadzących do ciężkich grzechów,&nbsp;</span> <span data-start="3321" data-end="3360" style="color: var(--vscode-foreground);">największa liczba nadzorowanych kar</span><span style="color: var(--vscode-foreground);">.&nbsp;</span> <span style="color: var(--vscode-foreground);">W każdej kategorii wybierani są najlepsi kandydaci (TOP 333), którzy zostają dodani do tabeli Czlonek_rady. W zwiazku z tym ich status diabła zmienia się na 'Członek rady', ich aktualne przydziały funkcji zostają zakończone (ponieważ w moim założeniu członek rady nie pełni innych funkcji), a wszystkie dotychczasowe nadzory kar zostają zamknięte.&nbsp;</span>

In [117]:
CREATE PROCEDURE UtworzRade
AS
BEGIN
    SET NOCOUNT ON;

    INSERT INTO Rada_piekielna
        (kadencja_od, kadencja_do)
    VALUES
        (GETDATE(), NULL);

    DECLARE @ID_rada INT;
    SET @ID_rada = SCOPE_IDENTITY(); -- Pobieram numer, który został nadany przez IDENTITY, dzięki temu mogę przypisać potem członków do rady

    DECLARE @Wybrani TABLE (ID_diabel INT PRIMARY KEY); -- Potrzebuję tutaj stworzyć tabelę tymczasową, aby zapisywać diabły, które wybieram, aby zabezpieczyć się przed sytuacją, w której jeden diabeł został wybrany kilka razy do tej samej rady, bo np. był top kusicielem i top kusicielem na najcięższe grzechy

    ;WITH
        TopKuszenia
        AS
        (
            SELECT TOP 333
                d.ID_diabel
            FROM Kuszenie k
                JOIN Diabel d ON k.kuszenie_ID_diabel = d.ID_diabel
                JOIN Osoba_zyjaca oz ON k.kuszenie_ID_osoba = oz.ID_osoba
            WHERE oz.czy_zyje = 1
                AND k.skutecznosc = 1
                AND d.status = N'Zwykły'
            GROUP BY d.ID_diabel
            ORDER BY COUNT(*) DESC
        )
    INSERT INTO Czlonek_rady
        (powod_wyboru, czlonek_ID_diabel, czlonek_ID_rada)
    SELECT
        N'Najwięcej skutecznych kuszeń',
        ID_diabel,
        @ID_rada
    FROM TopKuszenia;

    INSERT INTO @Wybrani
    SELECT ID_diabel
    FROM TopKuszenia;

    ;WITH
        Gorliwi
        AS
        (
            SELECT TOP 100
                modlitwa_ID_osobaZyjaca
            FROM Modlitwa
            GROUP BY modlitwa_ID_osobaZyjaca
            ORDER BY COUNT(*) DESC
        ),

        TopGorliwi
        AS
        (
            SELECT TOP 333
                d.ID_diabel
            FROM Kuszenie k
                JOIN Diabel d ON k.kuszenie_ID_diabel = d.ID_diabel
                JOIN Gorliwi g ON k.kuszenie_ID_osoba = g.modlitwa_ID_osobaZyjaca
            WHERE k.skutecznosc = 1
                AND d.status = N'Zwykły'
                AND d.ID_diabel NOT IN (SELECT ID_diabel
                FROM @Wybrani)
            GROUP BY d.ID_diabel
            ORDER BY COUNT(*) DESC
        )
    INSERT INTO Czlonek_rady
        (powod_wyboru, czlonek_ID_diabel, czlonek_ID_rada)
    SELECT
        N'Najwięcej kuszeń gorliwych modlących',
        ID_diabel,
        @ID_rada
    FROM TopGorliwi;

    INSERT INTO @Wybrani
    SELECT ID_diabel
    FROM TopGorliwi;

    ;WITH
        TopCiezkie
        AS
        (
            SELECT TOP 333
                d.ID_diabel
            FROM Kuszenie k
                JOIN Diabel d ON k.kuszenie_ID_diabel = d.ID_diabel
                JOIN Grzech g ON k.kuszenie_ID_grzech = g.ID_grzech
            WHERE g.kategoria = N'Ciężki'
                AND k.skutecznosc = 1
                AND d.status = N'Zwykły'
                AND d.ID_diabel NOT IN (SELECT ID_diabel
                FROM @Wybrani)
            GROUP BY d.ID_diabel
            ORDER BY COUNT(*) DESC
        )
    INSERT INTO Czlonek_rady
        (powod_wyboru, czlonek_ID_diabel, czlonek_ID_rada)
    SELECT
        N'Najwięcej kuszeń na ciężkie grzechy',
        ID_diabel,
        @ID_rada
    FROM TopCiezkie;

    INSERT INTO @Wybrani
    SELECT ID_diabel
    FROM TopCiezkie;

    ;WITH
        TopDreczyciele
        AS
        (
            SELECT TOP 333
                d.ID_diabel
            FROM Nadzor_kary nk
                JOIN Diabel d ON nk.nadzor_ID_diabel = d.ID_diabel
            WHERE d.status = N'Zwykły'
                AND d.ID_diabel NOT IN (SELECT ID_diabel
                FROM @Wybrani)
            GROUP BY d.ID_diabel
            ORDER BY COUNT(*) DESC
        )
    INSERT INTO Czlonek_rady
        (powod_wyboru, czlonek_ID_diabel, czlonek_ID_rada)
    SELECT
        N'Najwięcej nadzorowanych kar',
        ID_diabel,
        @ID_rada
    FROM TopDreczyciele;

    INSERT INTO @Wybrani
    SELECT ID_diabel
    FROM TopDreczyciele;

    -- Aktualizuję statusy diabłów na członków rady
    UPDATE Diabel
    SET status = N'Członek rady'
    WHERE ID_diabel IN (
        SELECT czlonek_ID_diabel
    FROM Czlonek_rady
    WHERE czlonek_ID_rada = @ID_rada
    );

    -- Kończę aktualne przydziały funkcji
    UPDATE Przydzial_funkcji
    SET data_do = GETDATE()
    WHERE data_do IS NULL
        AND przydzial_ID_diabel IN (
    SELECT czlonek_ID_diabel
        FROM Czlonek_rady
        WHERE czlonek_ID_rada = @ID_rada
  );

    -- Na sam koniec aktualizuje tabelę Nadzor_kary, bo członkowie rady według mojej logiki systemu piekielnego zostają awansowani i nie mogą już sprawować nadzoru nad karami
    UPDATE Nadzor_kary
    SET data_do = GETDATE()
    WHERE data_do IS NULL
        AND nadzor_ID_diabel IN (
        SELECT czlonek_ID_diabel
        FROM Czlonek_rady
        WHERE czlonek_ID_rada = @ID_rada
    );
    PRINT 'Nowa Rada Piekielna została utworzona.';
END;
GO

Commands completed successfully.

Total execution time: 00:00:00.051

Ostatnim i jednym z kluczowych elementów systemu piekielnego jest procedura _UtworzRaport,_ której zadaniem jest generowanie okresowych zestawień populacji piekła dla Lucyfera. Zgodnie z case study, raport jest wykonywany raz na 666 lat i <span style="color: var(--vscode-foreground);">&nbsp;zbiera najważniejsze dane, takie jak:&nbsp;</span> <span style="color: var(--vscode-foreground);">łączna liczba wszystkich diabłów,&nbsp;</span> <span style="color: var(--vscode-foreground);">liczba diabłów w rozbiciu na kategorie (zwykły, członek rady, doradca),&nbsp;</span> <span style="color: var(--vscode-foreground);">liczba grzeszników odbywających karę,&nbsp;</span> <span style="color: var(--vscode-foreground);">różnice liczności diabłów i grzeszników w stosunku do poprzedniego okresu rozliczeniowego.&nbsp;</span>

In [118]:
CREATE PROCEDURE UtworzRaport
AS
BEGIN
    SET NOCOUNT ON;
    DECLARE @data DATETIME = GETDATE();
    DECLARE @data_poprzedniego_raportu DATE;

    SELECT TOP 1
        @data_poprzedniego_raportu = data
    FROM Raport
    ORDER BY data DESC;

    IF @data_poprzedniego_raportu IS NOT NULL
        AND DATEDIFF(YEAR, @data_poprzedniego_raportu, @data) < 666
    BEGIN
        RAISERROR('Nie minęło jeszcze 666 lat od ostatniego raportu.', 16, 1);
        RETURN;
    END;

    DECLARE @poprzedni_diably INT = (
        SELECT TOP 1
        liczba_diablow_lacznie
    FROM Raport
    ORDER BY data DESC
        );

    DECLARE @poprzedni_grzesznicy INT = (
        SELECT TOP 1
        liczba_grzesznikow
    FROM Raport
    ORDER BY data DESC
        );

    DECLARE @diably INT = (SELECT COUNT(*)
    FROM Diabel);
    DECLARE @zwykle INT = (SELECT COUNT(*)
    FROM Diabel
    WHERE status = N'Zwykły');
    DECLARE @czlonkowie INT = (SELECT COUNT(*)
    FROM Diabel
    WHERE status = N'Członek rady');
    DECLARE @doradcy INT = (SELECT COUNT(*)
    FROM Diabel
    WHERE status = N'Doradca Lucyfera');
    DECLARE @grzesznicy INT = (SELECT COUNT(*)
    FROM Wymierzona_kara
    WHERE data_do IS NULL OR data_do >= @data);

    DECLARE @rada INT = (
        SELECT TOP 1
        ID_rada
    FROM Rada_piekielna
    WHERE @data >= kadencja_od
        AND (kadencja_do IS NULL OR @data <= kadencja_do)
    ORDER BY kadencja_od DESC
        );

    INSERT INTO Raport
        (
        data,
        liczba_diablow_lacznie,
        liczba_diablow_zwyklych,
        liczba_czlonkow_rady,
        liczba_doradcow,
        liczba_grzesznikow,
        zmiana_diablow,
        zmiana_grzesznikow,
        raport_ID_rada
        )
    VALUES
        (
            @data,
            @diably,
            @zwykle,
            @czlonkowie,
            @doradcy,
            @grzesznicy,
            CASE WHEN @poprzedni_diably IS NULL
            THEN NULL
            ELSE @diably - @poprzedni_diably END, -- Liczymy różnicę tylko wtedy, gdy istnieje poprzedni raport i pobraliśmy z niego dane o tym, ile było grzeszników, a ile diabłów
            CASE WHEN @poprzedni_grzesznicy IS NULL
            THEN NULL
            ELSE @grzesznicy - @poprzedni_grzesznicy END,
            @rada
    );
END;

Commands completed successfully.

Total execution time: 00:00:00.052